# Public historical notebook
Outputs, authentication metadata and personal artifact links have been removed for publication.
This is a historical code record, NOT a complete retraining kit. Do not Run All.
See the stage README and aggregate training history. Private datasets and model archives are not included.


# Colab arxiv nusxasi

Bu VS Code’da ko‘rsatish uchun saqlangan tarixiy notebook. Asl bajarilishning matnli chiqishlari saqlandi. Maxfiy tokenlar yashiriladi, HTML/rasm chiqishlari olinmaydi. Treningni laptopda Run All qilmang. Haqiqiy yangi trening uchun Colab va alohida run kerak.


# Real izohlar: Uzum + Commeta + Google Play

Boshlanish: tugagan **10k model, step=313**. Yangi run, **12 541 train, 1 epoch, 392 qadam**.
**Alohida Colab notebook, L4 GPU. Eski barcha treninglar to‘xtagan bo‘lsin.**
Dataset Silver, barcha targetlar inson tekshirgan Gold emas. Natija kafolatlanmaydi.
Oldingi model/run saqlanadi. Gemini API yo‘q; Colab GPU compute va ~22 GiB yangi Drive backup sarflanadi.
Validation 688, test 705, preflight-quarantine 21. Test avtomatik ishlatilmaydi.

## 1. Tayyor ZIPni yuklash

Katakni bajaring va `uznorm-train-real-v1.zip` faylni tanlang. Drive mount kerak emas.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, stat, subprocess, sys, tempfile, uuid, zipfile
from google.colab import files

ZIP = Path('/content/uznorm-train-real-v1.zip')
EXPECTED_SHA256 = 'db97078a0317bf5226057ea7842a9d3d03e74b1a9da24d86fa5a63bfa521261f'
if not ZIP.is_file():
    files.upload()
if not ZIP.is_file() or hashlib.sha256(ZIP.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise RuntimeError('To‘g‘ri uznorm-train-real-v1.zip faylni yuklang. SHA-256 mos emas.')
KIT = Path(tempfile.mkdtemp(prefix='uznorm-real-kit-', dir='/content'))
with zipfile.ZipFile(ZIP) as bundle:
    infos = bundle.infolist()
    if len(infos) > 200 or len({i.filename for i in infos}) != len(infos) or sum(i.file_size for i in infos) > 100*1024**2:
        raise RuntimeError('ZIP hajmi/tarkibi noto‘g‘ri.')
    for info in infos:
        name, part = info.filename, Path(info.filename)
        if part.is_absolute() or '..' in part.parts or ':' in name or chr(92) in name or info.is_dir() or stat.S_ISLNK(info.external_attr >> 16):
            raise RuntimeError('Xavfli ZIP yo‘li.')
        target = KIT / part
        target.parent.mkdir(parents=True, exist_ok=True)
        with bundle.open(info) as incoming, target.open('xb') as outgoing:
            shutil.copyfileobj(incoming, outgoing)
manifest = json.loads((KIT/'PACKAGE.json').read_text())
for name, expected in manifest['files'].items():
    if hashlib.sha256((KIT/name).read_bytes()).hexdigest() != expected:
        raise RuntimeError('Paket fayli hash xatosi: ' + name)
print('Kutubxonalar tayyorlanmoqda...', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(KIT/'requirements-colab.txt')], check=True)
ENV = os.environ.copy()
ENV.update(PYTHONPATH=str(KIT)+os.pathsep+str(KIT/'src'), USE_TF='0', USE_FLAX='0', PYTHONUNBUFFERED='1')
checked = subprocess.run([sys.executable, '-u', '-m', 'stagereal.runner', '--package', str(KIT), '--check-only'],
                         env=ENV, capture_output=True, text=True)
print(checked.stdout)
if checked.returncode:
    print(checked.stderr)
    raise RuntimeError('Paket tekshiruvi o‘tmadi; trening boshlanmadi.')
print('PAKET TAYYOR:', KIT)


## 2. Google hisob va W&B

Secrets → `WANDB_API_KEY` → shu notebook uchun Notebook access.
Oldingi 10k modelning backupi turgan **o‘sha Google hisob** bilan tasdiqlang.
Bu katak treningni boshlamaydi. API kalitini oddiy kodga yozmang.


In [ ]:
from google.colab import auth, userdata
auth.authenticate_user()
ENV = os.environ.copy()
ENV.update(PYTHONPATH=str(KIT)+os.pathsep+str(KIT/'src'), USE_TF='0', USE_FLAX='0', PYTHONUNBUFFERED='1')
try:
    key = userdata.get('WANDB_API_KEY').strip()
except Exception:
    raise RuntimeError('Colab Secrets: WANDB_API_KEY va Notebook access kerak.') from None
if not key:
    raise RuntimeError('WANDB_API_KEY bo‘sh.')
ENV['WANDB_API_KEY'] = key
del key
print('Ruxsatlar tayyor, key chiqarilmadi. Trening hali boshlanmadi.')


## 3. Boshlash / shu real-stage davom ettirish

`START_REAL_TRAINING = True` qiling va **shu katakni bajaring**.
Bu barcha eski treninglar to‘xtagani, Silver data, GPU sarfi va yangi Drive backuplarga tasdiq.
Ikki sessiyada parallel boshlamang. Eski model/running bayroqlarini o‘zgartirmang.

Loglar shu yerda ko‘rinadi: download → SHA-256 → GPU smoke → baseline → step/392.
Har 10 qadamda taxminiy training ETA, har 50 qadamda cloud tasdig‘i.
Backup paytida step turib qolishi normal; `STAGE_CLOUD_CONFIRMED`ni kuting.
Uzilsa **shu paket** bilan qayta kiring: real-stage checkpointi tiklanadi, eski modelga qaytmaydi.


In [ ]:
START_REAL_TRAINING = True
if START_REAL_TRAINING:
    import importlib.util
    spec = importlib.util.spec_from_file_location('real_notebook_support', KIT/'stagereal/notebook_support.py')
    support = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(support)
    command = [sys.executable, '-u', '-m', 'stagereal.runner', '--package', str(KIT),
               '--allow-cloud', '--old-stopped', '--allow-non-gold']
    LOG = Path('/content') / ('real-training-' + uuid.uuid4().hex[:12] + '.log')
    support.run_visible(command, ENV, LOG)
else:
    print('Boshlash uchun START_REAL_TRAINING=True qilib shu katakni bajaring.')


In [ ]:
from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

# 1. Model saqlangan manzil
model_path = "/content/uznorm-real-vtefitvr/run/checkpoint-392"

# 2. Tokenizator va modelni yuklash
print("Model yuklanmoqda...")
tokenizer = AutoTokenizer.from_pretrained(model_path)

# from_pt=True opsiyasi orqali agar checkpoint PyTorch formatida
# saqlangan bo'lsa ham uni to'g'ridan-to'g'ri TensorFlow'ga o'girib olamiz.
model = TFAutoModelForSeq2SeqLM.from_pretrained(model_path, from_pt=True)
print("Model tayyor!")

# 3. Tuzatish funksiyasi
def matnni_tuzatish(matn):
    # Kiruvchi matnni UTF-8 baytlarga ajratib, tenzorga o'giramiz
    inputs = tokenizer(matn, return_tensors="tf")

    # Model orqali xatosiz matnni generatsiya qilamiz
    outputs = model.generate(
        **inputs,
        max_length=128,      # maksimal uzunlik
        num_beams=5,         # beam search - eng ehtimolli variantni qidirish
        early_stopping=True
    )

    # Natijaviy baytlarni oddiy inson o'qiydigan matnga qaytaramiz
    to_g_rilangan_matn = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return to_g_rilangan_matn

# 4. Sinov uchun matnlar
test_matnlar = [
    "salom qaleysiz manga tezroq telifon ql",
    "Uzum dan narsa zkaz qildim, zo'r ekan",
    "hozr kelaman, ozro kutp turing"
]

print("-" * 40)
for matn in test_matnlar:
    natija = matnni_tuzatish(matn)
    print(f"Xato matn  : {matn}")
    print(f"Tuzatilgan : {natija}")
    print("-" * 40)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Interaktiv oynalar yaratish
matn_kiritish = widgets.Textarea(
    value='',
    placeholder='Xato matnni shu yerga yozing...',
    description='Matn:',
    disabled=False,
    layout=widgets.Layout(width='80%', height='100px')
)

tugma = widgets.Button(
    description='To\'g\'rilash',
    button_style='success',
    tooltip='Matnni to\'g\'rilash uchun bosing'
)

natija_chiqarish = widgets.Output()

def tugma_bosilganda(b):
    with natija_chiqarish:
        clear_output() # Oldingi natijani tozalash
        kiritilgan_matn = matn_kiritish.value
        if not kiritilgan_matn.strip():
            print("Iltimos, matn kiriting!")
            return

        print("Model ishlayapti...")

        # Kiruvchi matnni UTF-8 baytlarga ajratib, tenzorga o'giramiz
        inputs = tokenizer(kiritilgan_matn, return_tensors="tf")

        # Model orqali xatosiz matnni generatsiya qilamiz
        outputs = model.generate(
            **inputs,
            max_length=736,
            num_beams=5,
            length_penalty=2.0,
            early_stopping=True
        )

        # Natijaviy baytlarni oddiy inson o'qiydigan matnga qaytaramiz
        to_g_rilangan_matn = tokenizer.decode(outputs[0], skip_special_tokens=True)

        clear_output()
        print("-" * 50)
        print(f"Siz kiritdingiz : {kiritilgan_matn}")
        print(f"Model to'g'riladi: {to_g_rilangan_matn}")
        print("-" * 50)

tugma.on_click(tugma_bosilganda)

# Ekranga chiqarish
display(matn_kiritish, tugma, natija_chiqarish)

In [ ]:
!pip install jiwer scikit-learn

In [ ]:
import jiwer
import re
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score

# --- DIQQAT: tokenizer va model avvalgi kataklarda yuklangan bo'lishi kerak ---
# Agar yuklanmagan bo'lsa, ularni shu yerda yuklab oling.

# 2. Sizning baholash dataset (Gold Dataset)
# Bu yerga 50-100 ta uzun va xatoli izohlaringizni kiriting
ma_lumotlar = [
    {
        "xato": "salom man 2ta krasovka zkaz qildim oqi tor kankretniy qisib turibdi rassiyski razmerga togri kemaskan kimmandir shunaqa obmen qildimi manga aytvorila yordamila kere raxmat kattakon",
        "gold": "Salom, men 2 ta krossovka zakaz qildim. Oqi tor, konkretniy qisib turibdi. Rossiyskiy razmerga to'g'ri kelmas ekan. Kimdadir shunaqa, obmen qildimi? Menga aytib yuboringlar, yordaminglar kerak, rahmat kattakon."
    },
    {
        "xato": "maxsulot gap yoq menga raxmati hammaga tafsiya qilaman karopkasi chiroyli qadoglanibdi upakovkasi umuman zor oylaganimdan ham yaxshi chikdi 5 yulduz kam bunga",
        "gold": "Mahsulot gap yo'q, menga yoqdi, hammaga tavsiya qilaman. Karobkasi chiroyli qadoqlanibdi, upakovkasi umuman zo'r. O'ylaganimdan ham yaxshi chiqdi, 5 yulduz kam bunga."
    }
    # Yana matnlarni shu yerga qo'shib ketaverasiz...
]

xato_matnlar = [d["xato"] for d in ma_lumotlar]
gold_matnlar = [d["gold"] for d in ma_lumotlar]
pred_matnlar = []

print(f"Jami {len(xato_matnlar)} ta matn model orqali to'g'rilanmoqda...")

# 3. Model orqali generatsiya qilish (Erta to'xtashning oldini olish parametrlari bilan)
for xato in tqdm(xato_matnlar, desc="Generatsiya qilinmoqda"):
    inputs = tokenizer(xato, return_tensors="tf")
    outputs = model.generate(
        **inputs,
        max_length=512,        # Uzun matnlar uchun chegarani kengaytiramiz
        num_beams=5,
        length_penalty=1.5,    # Modelni uzunroq matn yaratishga undash
        early_stopping=False   # Erta to'xtab qolishning oldini olish
    )
    to_g_rilangan = tokenizer.decode(outputs[0], skip_special_tokens=True)
    pred_matnlar.append(to_g_rilangan)

print("\nBarcha matnlar generatsiya qilindi. Baholash boshlanmoqda...\n")

# 4. Metrikalarni hisoblash funksiyalari
def tozalash(matn):
    return re.sub(r'[^\w\s]', '', matn.lower())

def apostrof_sanamoq(matn):
    return len(re.findall(r"[oO0][\'\‘\’\`]+|[gG][\'\‘\’\`]+|[a-zA-Z][\'\‘\’\`]+[a-zA-Z]", matn))

gold_toza = [tozalash(m) for m in gold_matnlar]
xato_toza = [tozalash(m) for m in xato_matnlar]
pred_toza = [tozalash(m) for m in pred_matnlar]

# WER va CER hisoblash
baseline_wer = jiwer.wer(gold_toza, xato_toza)
model_wer = jiwer.wer(gold_toza, pred_toza)

baseline_cer = jiwer.cer(gold_toza, xato_toza)
model_cer = jiwer.cer(gold_toza, pred_toza)

# Apostrof hisoblash
gold_apos = [apostrof_sanamoq(m) for m in gold_matnlar]
xato_apos = [apostrof_sanamoq(m) for m in xato_matnlar]
pred_apos = [apostrof_sanamoq(m) for m in pred_matnlar]

baseline_apos_acc = accuracy_score(gold_apos, xato_apos)
model_apos_acc = accuracy_score(gold_apos, pred_apos)

# 5. Natijalarni chiroyli formatda chiqarish
print("="*60)
print(f"{'METRIKA':<20} | {'BOSHLANG\'ICH (Baseline)':<22} | {'MODEL (ByT5)':<15}")
print("-" * 60)
print(f"{'Content WER':<20} | {baseline_wer:<22.1%} | {model_wer:<15.1%}")
print(f"{'Spelling CER':<20} | {baseline_cer:<22.1%} | {model_cer:<15.1%}")
print(f"{'Apostrof Mosligi':<20} | {baseline_apos_acc:<22.1%} | {model_apos_acc:<15.1%}")
print("="*60)

print("\nNAMUNALAR SOLISHTIRUVI:")
for i in range(min(2, len(ma_lumotlar))):  # Faqat dastlabki 2 tasini ko'rsatish
    print(f"\n[{i+1}] XATO : {xato_matnlar[i]}")
    print(f"[{i+1}] PRED : {pred_matnlar[i]}")
    print(f"[{i+1}] GOLD : {gold_matnlar[i]}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip "/content/uznorm-train-real-v1.zip" -d "/content/model_papka"

In [ ]:
model_path = "/content/model_papka/checkpoint-392" # Zip ochilgan joy

In [ ]:
from google.colab import drive
import os

# 1. Google Drive'ni ulash (ruxsat so'raydi)
drive.mount('/content/drive')

# 2. Drive ichidan modelni qidirish
print("\nGoogle Drive ichidan model qidirilmoqda...")
topildi = False

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    # Papkalarni qidirish
    for name in dirs:
        if 'checkpoint-392' in name or 'uznorm-real' in name:
            print(f"✅ Topildi (Papka): {os.path.join(root, name)}")
            topildi = True

    # Zip arxivlarni qidirish (agar arxivlab saqlangan bo'lsa)
    for name in files:
        if 'checkpoint-392' in name or 'uznorm' in name.lower():
            if name.endswith('.zip'):
                print(f"📦 Topildi (Zip fayl): {os.path.join(root, name)}")
                topildi = True

if not topildi:
    print("❌ Hech narsa topilmadi. Tizim uni W&B ga yuklagan bo'lishi mumkin.")


In [ ]:
!unzip -q "/content/drive/MyDrive/uznorm/uznorm-cloud-ureal-14a56cf24f5b1116/checkpoint-392-b3b4c95291cf35648fc96bee.zip" -d "/content/model_checkpoint"
print("Model arxivi muvaffaqiyatli ochildi!")

In [ ]:
!pip install -q sentencepiece

In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Haqiqiy model papkasini topish (og'irliklari bor papka)
base_dir = "/content/model_checkpoint"
aniq_manzil = None

for root, dirs, files in os.walk(base_dir):
    # Agar papkada ham config.json, ham model og'irliklari bo'lsa:
    if "config.json" in files and any(f in files for f in ["model.safetensors", "pytorch_model.bin", "tf_model.h5"]):
        aniq_manzil = root
        break

if not aniq_manzil:
    print("❌ Haqiqiy model papkasi topilmadi! Arxiv ichidagi fayllar ro'yxati:")
    os.system(f"find {base_dir}")
    raise FileNotFoundError("Model og'irliklari topilmadi.")

print(f"✅ Haqiqiy model shu manzildan o'qilmoqda: {aniq_manzil}")

# 2. Tokenizator (standart) va Modelni yuklash
tokenizer = AutoTokenizer.from_pretrained("google/byt5-base")
# Agar model TensorFlow bo'lsa, from_pt=True bilan o'qimaymiz.
# Avvalgi loglarda TF deyilgan, shuning uchun buni try-except qilib xavfsiz yuklaymiz
try:
    model = AutoModelForSeq2SeqLM.from_pretrained(aniq_manzil)
except:
    model = AutoModelForSeq2SeqLM.from_pretrained(aniq_manzil, from_pt=True)

print("🚀 ByT5 Model tayyor!\n")

# 3. Interaktiv oynani yaratish
matn_kiritish = widgets.Textarea(
    value='',
    placeholder='E-commerce izohini shu yerga yozing...',
    description='Matn:',
    layout=widgets.Layout(width='80%', height='100px')
)
tugma = widgets.Button(description="To'g'rilash", button_style='success')
natija_chiqarish = widgets.Output()

def tugma_bosilganda(b):
    with natija_chiqarish:
        clear_output()
        matn = matn_kiritish.value
        if not matn.strip(): return

        print("Model hisoblamoqda... Kutib turing.")

        inputs = tokenizer(matn, return_tensors="pt")

        outputs = model.generate(
            **inputs,
            max_length=512,
            num_beams=5,
            length_penalty=1.5,
            early_stopping=False
        )

        tuzatilgan = tokenizer.decode(outputs[0], skip_special_tokens=True)

        clear_output()
        print("-" * 50)
        print(f"Xom Matn     : {matn}")
        print(f"To'g'rilangan: {tuzatilgan}")
        print("-" * 50)

tugma.on_click(tugma_bosilganda)
display(matn_kiritish, tugma, natija_chiqarish)

## Natija

**`REAL_STAGE_COMPLETE_CLOUD_VERIFIED step=392`** — trening, oldin/keyin baholash,
yakuniy cloud upload va qayta yuklab tekshirish tugadi.
Oxirgi `COMPARISON:` yo‘lidagi faylni oching: real va synthetic ko‘rsatkichlar alohida.
`MODEL:` yangi sinov modeli joylashgan papka. Eski modelni yaxshilanish tasdiqlanmaguncha almashtirmang.
Bu 144 real + 128 synthetic development baholash; ma’no to‘g‘riligi/Gold accuracy testi emas.
